# pop execution-eval -- one-notebook Colab run (arms A, C, D over the 201 bugs)

Runs **Step 4** of the study: feed each bug-fixing arm's generator over the 201 vendored
QuixBugs-Java (40) + HumanEval-Java (161) bugs, then score the candidate fixes with the JDK
compile-and-run harness. Produces `results/execbench_A.json` (pretrained->finetuned T5),
`results/execbench_C.json` (RAG-Qwen, best config), and `results/execbench_D.json` (LoRA-Qwen)
-- the pass@1 / compile-rate numbers behind the report's execution-vs-CodeBLEU lens (Figure 3).

**Prerequisites (all on Drive `MyDrive/pop_cycle3/` already, from the other notebooks):**
- the shared `pop_repo.zip` (same one the RAG/scaling/LoRA notebooks use -- no code changed, so
  the existing upload works; just upload *this* notebook),
- `outputs/pretrain/final` + `outputs/tokenizer/tokenizer.model` (from the scaling notebook's
  Step 0) -- used to finetune arm A's T5 if it isn't on Drive yet,
- `results/rag_*_test.json` (8 files, from the RAG notebook) -- picks arm C's best config,
- `outputs/lora_qwen/best` (from the LoRA notebook) -- arm D's adapter.

**Every session**: Runtime > Change runtime type > pick a **GPU**, then Runtime > **Run all**.
Generation and scoring cells skip any output already on Drive, so re-running after a disconnect
continues where it left off.

> **Heads-up on arm A (~1 h once):** the execution lens needs a finetuned arm-A T5
> (`outputs/finetune_A_ep10/best`). No earlier notebook produces it, so the arm-A cell below
> finetunes it from the Step-0 pretrain the first time (seed 42, ~1 h on A100). It auto-resumes
> from checkpoints and skips entirely once `best/` exists. Arms C and D are quick (~5-10 min each).

In [ ]:
import sys

print("Python", sys.version)
assert sys.version_info >= (3, 11), "pop needs Python >= 3.11; this Colab runtime is older"
!nvidia-smi

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/pop_cycle3")
for sub in ("outputs", "results", "logs"):
    (BASE / sub).mkdir(parents=True, exist_ok=True)

ZIP = BASE / "pop_repo.zip"
assert ZIP.is_file(), (
    f"Upload pop_repo.zip to {ZIP.parent} first -- build it locally with\n"
    "  git archive --format=zip -o dist/pop_repo.zip HEAD\n"
    "then drag dist/pop_repo.zip into Drive/pop_cycle3/ (see docs/colab-runbook.md)."
)
print("Drive workspace ready:", BASE)

In [ ]:
%cd /content
!rm -rf /content/repo
!unzip -q /content/drive/MyDrive/pop_cycle3/pop_repo.zip -d /content/repo
%cd /content/repo
%pip install -q -e .

In [ ]:
# Point the repo's outputs/, results/, logs/ at Drive so predictions, metrics, and any
# arm-A finetune checkpoints all survive session resets. The repo ships a committed results/
# directory; its contents are copied onto Drive once before the swap.
import os
import shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/pop_cycle3")
REPO = Path("/content/repo")
for sub in ("outputs", "results", "logs"):
    drive_dir = BASE / sub
    repo_dir = REPO / sub
    if repo_dir.is_symlink():
        repo_dir.unlink()
    elif repo_dir.exists():
        shutil.copytree(repo_dir, drive_dir, dirs_exist_ok=True)
        shutil.rmtree(repo_dir)
    os.symlink(drive_dir, repo_dir)
print("outputs/, results/, logs/ now live on Drive:", BASE)

### Environment fix: remove the stale torchao

Colab ships an old **torchao 0.10**, which the installed `peft` hard-rejects when loading a LoRA
adapter (`Found an incompatible version of torchao ...`). Arm D reads a plain-LoRA adapter (no
torchao quantization), so uninstalling torchao lets `peft` skip that code path. Without this cell,
arm D's generation crashes on adapter load. (Harmless for arms A/C, which don't touch peft.)

In [ ]:
# The gen script runs in a subprocess (`!python ...`), which picks up this uninstall. Harmless
# if torchao is already absent (pip just skips it).
!pip uninstall -y -q torchao

### JDK for the scoring harness

`pop execbench` compiles and runs each candidate with `javac`/`java`. The 201-bug manifest is
**curated to modern JDKs** (`scripts/build_benchmark_manifests.py` excludes the Nashorn/JAXB bugs
that need APIs removed after JDK 8), and the reference set validates 201/201 on JDK 21. We install
**OpenJDK 17** and pass its home to `--jdk` so scoring is deterministic regardless of Colab's
default `java`. Generation (above) needs no JDK; only the score cells use it.

In [ ]:
# Install a modern JDK and pin its home (deterministic, reference-matching toolchain).
import glob
import os

!apt-get -qq install -y openjdk-17-jdk-headless
homes = sorted(glob.glob("/usr/lib/jvm/java-17-openjdk*"))
assert homes, "openjdk-17 install produced no java-17 home under /usr/lib/jvm"
JDK_HOME = homes[0]
os.environ["JDK_HOME"] = JDK_HOME
print("JDK_HOME =", JDK_HOME)
!{JDK_HOME}/bin/java -version
!{JDK_HOME}/bin/javac -version

## Arm A -- pretrained -> finetuned T5

Needs a finetuned arm-A T5 at `outputs/finetune_A_ep10/best`. The cell below finetunes it from
the Step-0 pretrain (`outputs/pretrain/final`, on Drive from the scaling notebook) **only if it
isn't there yet** (~1 h on A100, auto-resumes from checkpoints). Then it generates a candidate for
each buggy *file* and scores them. Expect **low compile rates**: every arm was trained on single
*methods* but is fed whole buggy *files* here -- that whole-file-vs-method mismatch is itself the
Track-2 finding (see the header of `scripts/gen_execbench_predictions.py`), not a bug.

In [ ]:
# Finetune arm A's T5 only if it's not already on Drive (idempotent; skips once best/ exists).
from pathlib import Path

if (Path("outputs/finetune_A_ep10/best") / "config.json").exists():
    print("arm A T5 already on Drive (outputs/finetune_A_ep10/best) -- skipping finetune")
else:
    print("arm A T5 not found -- finetuning ep10 from the Step-0 pretrain (~1 h on A100)...")
    !pop finetune --config configs/finetune_A_ep10.yaml

In [ ]:
# Arm A generate over QuixBugs (skips if the jsonl is already on Drive).
import os

out = "outputs/execbench/A_quixbugs.jsonl"
if os.path.exists(out):
    print(out, "exists -- skipping (delete to regenerate)")
else:
    !python scripts/gen_execbench_predictions.py --arm t5 --model outputs/finetune_A_ep10/best --tokenizer outputs/tokenizer/tokenizer.model --bench quixbugs --out {out}

In [ ]:
# Arm A generate over HumanEval-Java (separate cell so each bench's jsonl persists for resume).
import os

out = "outputs/execbench/A_humaneval_java.jsonl"
if os.path.exists(out):
    print(out, "exists -- skipping (delete to regenerate)")
else:
    !python scripts/gen_execbench_predictions.py --arm t5 --model outputs/finetune_A_ep10/best --tokenizer outputs/tokenizer/tokenizer.model --bench humaneval_java --out {out}

In [ ]:
# Concatenate both benches and score once (each record carries its own 'bench'). Skips if done.
import os

res = "results/execbench_A.json"
if os.path.exists(res):
    print(res, "exists -- skipping scoring (delete to re-score)")
else:
    !cat outputs/execbench/A_quixbugs.jsonl outputs/execbench/A_humaneval_java.jsonl > outputs/execbench/A_all.jsonl
    !pop execbench --predictions outputs/execbench/A_all.jsonl --name execbench_A --jdk {JDK_HOME} --jobs 4

## Arm C -- RAG Qwen (best config)

Picks the highest-CodeBLEU config from the RAG notebook's 8 `results/rag_*_test.json`, rebuilds
that retriever's KB from the **train** split (leakage guard), prompts Qwen over the 201 buggy
files, and scores. If the best config is `k0`, arm C is effectively the zero-shot baseline -- that
is a valid, principled choice of "best RAG arm".

In [ ]:
# Choose arm C = the RAG config with the highest CodeBLEU on the test split.
import glob
import json
import os

best = None
for p in sorted(glob.glob("results/rag_*_test.json")):
    codebleu = json.loads(open(p, encoding="utf-8").read())["metrics"]["codebleu"]
    name = os.path.basename(p).replace("_test.json", "")
    if best is None or codebleu > best[1]:
        best = (name, codebleu)
assert best, "no results/rag_*_test.json found -- run the RAG notebook first"
BEST_RAG_CFG = f"configs/{best[0]}.yaml"
assert os.path.exists(BEST_RAG_CFG), f"config for best RAG arm not found: {BEST_RAG_CFG}"
print(f"Best RAG arm by CodeBLEU: {best[0]} (CodeBLEU={best[1]:.4f})  ->  {BEST_RAG_CFG}")

In [ ]:
import os

out = "outputs/execbench/C_quixbugs.jsonl"
if os.path.exists(out):
    print(out, "exists -- skipping (delete to regenerate)")
else:
    !python scripts/gen_execbench_predictions.py --arm rag --config {BEST_RAG_CFG} --bench quixbugs --out {out}

In [ ]:
import os

out = "outputs/execbench/C_humaneval_java.jsonl"
if os.path.exists(out):
    print(out, "exists -- skipping (delete to regenerate)")
else:
    !python scripts/gen_execbench_predictions.py --arm rag --config {BEST_RAG_CFG} --bench humaneval_java --out {out}

In [ ]:
import os

res = "results/execbench_C.json"
if os.path.exists(res):
    print(res, "exists -- skipping scoring (delete to re-score)")
else:
    !cat outputs/execbench/C_quixbugs.jsonl outputs/execbench/C_humaneval_java.jsonl > outputs/execbench/C_all.jsonl
    !pop execbench --predictions outputs/execbench/C_all.jsonl --name execbench_C --jdk {JDK_HOME} --jobs 4

## Arm D -- LoRA Qwen

Reads the adapter from `outputs/lora_qwen/best` (from the LoRA notebook), builds the *same*
instruction+buggy prompt the adapter was trained on (shared `build_lora_prompt`), generates over
the 201 buggy files, and scores.

In [ ]:
import os

out = "outputs/execbench/D_quixbugs.jsonl"
if os.path.exists(out):
    print(out, "exists -- skipping (delete to regenerate)")
else:
    !python scripts/gen_execbench_predictions.py --arm lora --config configs/lora_qwen.yaml --bench quixbugs --out {out}

In [ ]:
import os

out = "outputs/execbench/D_humaneval_java.jsonl"
if os.path.exists(out):
    print(out, "exists -- skipping (delete to regenerate)")
else:
    !python scripts/gen_execbench_predictions.py --arm lora --config configs/lora_qwen.yaml --bench humaneval_java --out {out}

In [ ]:
import os

res = "results/execbench_D.json"
if os.path.exists(res):
    print(res, "exists -- skipping scoring (delete to re-score)")
else:
    !cat outputs/execbench/D_quixbugs.jsonl outputs/execbench/D_humaneval_java.jsonl > outputs/execbench/D_all.jsonl
    !pop execbench --predictions outputs/execbench/D_all.jsonl --name execbench_D --jdk {JDK_HOME} --jobs 4

### Arm B — from-scratch T5 (optional execution point)

Arm B has near-identical CodeBLEU to arm A, so its execution point is **optional** — the report's
Track-2 story stands without it. Running the four cells below closes the one remaining hedge in the
report (*"arm B was not run through the harness … an expectation, not a measurement"*) by producing
`results/execbench_B.json`.

Arm B is finetuned **from a random init** (`configs/finetune_B_seed0.yaml` deliberately omits
`pretrained_model_path`, so `pop finetune` builds a fresh model — no pretrain checkpoint needed); it
reuses the same tokenizer as arm A. Expect the **same low compile rate** as arm A (the
whole-file-vs-method mismatch), which is exactly the Track-2 point.

In [ ]:
# Finetune arm B (from-scratch, seed 0) only if it's not already on Drive (idempotent; skips once
# best/ exists). No pretrain checkpoint is used -- finetune_B_seed0.yaml omits pretrained_model_path.
from pathlib import Path

if (Path("outputs/finetune_B_seed0/best") / "config.json").exists():
    print("arm B T5 already on Drive (outputs/finetune_B_seed0/best) -- skipping finetune")
else:
    print("arm B T5 not found -- finetuning from-scratch seed 0 (~1 h on A100, auto-resumes)...")
    !pop finetune --config configs/finetune_B_seed0.yaml

In [ ]:
# Arm B generate over QuixBugs (skips if the jsonl is already on Drive).
import os

out = "outputs/execbench/B_quixbugs.jsonl"
if os.path.exists(out):
    print(out, "exists -- skipping (delete to regenerate)")
else:
    !python scripts/gen_execbench_predictions.py --arm t5 --model outputs/finetune_B_seed0/best --tokenizer outputs/tokenizer/tokenizer.model --bench quixbugs --out {out}

In [ ]:
# Arm B generate over HumanEval-Java (separate cell so each bench's jsonl persists for resume).
import os

out = "outputs/execbench/B_humaneval_java.jsonl"
if os.path.exists(out):
    print(out, "exists -- skipping (delete to regenerate)")
else:
    !python scripts/gen_execbench_predictions.py --arm t5 --model outputs/finetune_B_seed0/best --tokenizer outputs/tokenizer/tokenizer.model --bench humaneval_java --out {out}

In [ ]:
# Concatenate both benches and score once (each record carries its own 'bench'). Skips if done.
import os

res = "results/execbench_B.json"
if os.path.exists(res):
    print(res, "exists -- skipping scoring (delete to re-score)")
else:
    !cat outputs/execbench/B_quixbugs.jsonl outputs/execbench/B_humaneval_java.jsonl > outputs/execbench/B_all.jsonl
    !pop execbench --predictions outputs/execbench/B_all.jsonl --name execbench_B --jdk {JDK_HOME} --jobs 4

In [ ]:
# Summary: pass@1 / compile-rate per arm (+ per-benchmark breakdown).
import json
import os

print("=== Execution-eval results ===")
for name in ("execbench_A", "execbench_C", "execbench_D", "execbench_B"):
    p = f"results/{name}.json"
    if not os.path.exists(p):
        if name != "execbench_B":
            print(f"{name}: (missing -- run its cells above)")
        continue
    m = json.loads(open(p, encoding="utf-8").read())["metrics"]
    pb = m.get("per_benchmark", {})
    qb, he = pb.get("quixbugs", {}), pb.get("humaneval_java", {})
    print(
        f"{name}: pass_rate={m['pass_rate']:.4f} compile_rate={m['compile_rate']:.4f} n={m['n']}  "
        f"[quixbugs pass={qb.get('pass_rate', 0):.3f}/n={qb.get('n', 0)}, "
        f"humaneval pass={he.get('pass_rate', 0):.3f}/n={he.get('n', 0)}]"
    )

### After this notebook

The `results/execbench_{A,C,D}.json` (plus `execbench_B.json` if you ran the optional arm-B cells)
now live under `Drive/pop_cycle3/results/` alongside the RAG/scaling/LoRA JSONs. **Aggregation runs
locally**, not here: download `results/` back to your clone, then a single
`python scripts/figures/make_all.py` rebuilds the derived CSVs from the JSONs and re-renders Figure 3
from the real numbers (arm B's point appears automatically once `execbench_B.json` is present).

**If you ran arm B:** fold the measured `execbench_B.json` number into `docs/report.md` — replace the
"arm B was not run … an expectation, not a measurement" hedges (Method §, findings 5/7, Limitations)
with the real pass@1, and rerun `make_all.py` so Figure 3 shows the fourth point.

**If the session disconnects:** reopen and **Run all**. Every generate/score cell skips its output if
it already exists on Drive, and each arm's finetune resumes from its latest checkpoint, so you only
redo what's missing.